In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/ipo_master_raw.csv')
print(f'Raw rows: {len(df)}')
print(df.head())
print(df.columns.tolist())

Raw rows: 2400
       IPO Date Symbol                             Company Name IPO Price  \
0  Mar 12, 2026   PAYP                       PayPay Corporation    $16.00   
1  Mar 11, 2026   SUMA             SUMA Acquisition Corporation    $10.00   
2   Mar 6, 2026   MMED                      Minimed Group, Inc.    $20.00   
3   Mar 4, 2026   GLED       GalaxyEdge Acquisition Corporation    $10.00   
4   Mar 4, 2026   KCAC  Kensington Capital Acquisition Corp. VI    $10.00   

  Current   Return  year  
0  $21.14   32.13%  2012  
1   $9.98   -0.20%  2012  
2  $15.76  -21.20%  2012  
3  $10.00        -  2012  
4  $10.05    0.50%  2012  
['IPO Date', 'Symbol', 'Company Name', 'IPO Price', 'Current', 'Return', 'year']


In [3]:
df['IPO Date'] = pd.to_datetime(df['IPO Date'], errors='coerce')
df['actual_year'] = df['IPO Date'].dt.year
print(df.groupby('actual_year').size())

actual_year
2025    1488
2026     912
dtype: int64


In [4]:
import requests

headers = {'User-Agent': 'Mozilla/5.0 (academic research project, UC Davis MSBA)'}

# test a few URL patterns for 2018
urls = [
    'https://stockanalysis.com/ipos/2018/',
    'https://stockanalysis.com/ipos/statistics/2018/',
    'https://stockanalysis.com/ipos/2018/all/',
]

for url in urls:
    r = requests.get(url, headers=headers)
    print(f'{r.status_code} -- {url}')

404 -- https://stockanalysis.com/ipos/2018/
404 -- https://stockanalysis.com/ipos/statistics/2018/
404 -- https://stockanalysis.com/ipos/2018/all/


In [5]:
import requests

headers = {'User-Agent': 'Mozilla/5.0 (academic research project, UC Davis MSBA)'}

for year in [2024, 2025]:
    url = f'https://stockanalysis.com/ipos/{year}/'
    r = requests.get(url, headers=headers)
    print(f'{r.status_code} -- {year}')

200 -- 2024
200 -- 2025


In [6]:
# Written by V
import pandas as pd

df = pd.read_csv('../data/raw/ipo_master_raw.csv')
df['IPO Date'] = pd.to_datetime(df['IPO Date'], errors='coerce')

print(f'Before cleaning: {len(df)} rows')

# remove SPACs
spac_keywords = ['acquisition', 'blank check', 'spac']
mask_spac = df['Company Name'].str.lower().str.contains('|'.join(spac_keywords), na=False)
df = df[~mask_spac]
print(f'After removing SPACs: {len(df)} rows')

# remove rows with no valid IPO price
df = df[df['IPO Price'] != '-']
df = df[df['IPO Price'].notna()]
print(f'After removing no-price rows: {len(df)} rows')

# drop rows where IPO date is null
df = df[df['IPO Date'].notna()]
print(f'After removing null dates: {len(df)} rows')

# rename columns to match schema
df = df.rename(columns={
    'Symbol': 'ticker',
    'Company Name': 'company_name',
    'IPO Date': 'ipo_date',
})

df = df[['ticker','company_name', 'ipo_date', 'year']]
df.to_csv('../data/cleaned/ipo_master.csv', index=False)
print(f'\nFinal: {len(df)} rows saved to data/cleaned/ipo_master.csv')

Before cleaning: 2655 rows
After removing SPACs: 1723 rows
After removing no-price rows: 1675 rows
After removing null dates: 1675 rows

Final: 1675 rows saved to data/cleaned/ipo_master.csv


In [8]:
import yfinance as yf
import pandas as pd
import time

df = pd.read_csv('../data/cleaned/ipo_master.csv')
print(f'Starting validation: {len(df)} tickers')

missing = []

for i, row in df.iterrows():
    ticker = row['ticker']
    try:
        data = yf.download(ticker, period='1mo', progress=False, auto_adjust=True)
        if data.empty:
            missing.append(ticker)
    except Exception as e:
        missing.append(ticker)
    
    if i % 50 == 0:
        print(f'Progress: {i}/{len(df)} -- missing so far: {len(missing)}')
    
    time.sleep(0.3)

print(f'\nDone. Missing tickers: {len(missing)}')
print(missing)

Starting validation: 1675 tickers
Progress: 0/1675 -- missing so far: 0


$CIIC: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CIIC']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$HCCO: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HCCO']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$OCFT: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['OCFT']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CHPM: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CHPM']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbo

Progress: 50/1675 -- missing so far: 15


$EXPC: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['EXPC']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$STSA: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['STSA']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$SWTX: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['SWTX']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CFB: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CFB']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol 

Progress: 100/1675 -- missing so far: 36


...

Progress: 1450/1675 -- missing so far: 474
Progress: 1500/1675 -- missing so far: 474
Progress: 1550/1675 -- missing so far: 474


$CCCM: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CCCM']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$CCCX: possibly delisted; no price data found  (period=1mo)

1 Failed download:
['CCCX']: possibly delisted; no price data found  (period=1mo)


Progress: 1600/1675 -- missing so far: 476
Progress: 1650/1675 -- missing so far: 476


$MTSR: possibly delisted; no price data found  (period=1mo)

1 Failed download:
['MTSR']: possibly delisted; no price data found  (period=1mo)



Done. Missing tickers: 477
['CIIC', 'HCCO', 'OCFT', 'CHPM', 'ETNB', 'JIH', 'MOHOY', 'CNTG', 'SICP', 'MCMJ', 'OYST', 'TFFP', 'VIE', 'PING', 'IGMS', 'EXPC', 'STSA', 'SWTX', 'CFB', 'CCX', 'WSG', 'FLLC', 'LVGO', 'NOVA', 'PROS', 'MDLA', 'AMK', 'PIC', 'SCPE', 'THCA', 'KRTX', 'CHNG', 'MORF', 'LINX', 'AKRO', 'BCEL', 'PRVL', 'SMMC', 'WORK', 'MWK', 'GIX', 'RTLR', 'AGBA', 'APLT', 'AXLA', 'HHR', 'LCA', 'ATIF', 'SCPL', 'MNRL', 'TPTX', 'TUFN', 'NGM', 'SILK', 'RUHN', 'GNFT', 'SWAV', 'THCB', 'DPHC', 'SOLY', 'MITO', 'AVDR', 'TCRR', 'ANCN', 'HARP', 'GMHI', 'MDJH', 'GBS', 'VII', 'MDWT', 'SCOA', 'VIRI', 'CCV', 'KINZ', 'GHVI', 'MOTV', 'PCPC', 'PTIC', 'SBTX', 'SGTX', 'HTPA', 'KNTE', 'CAP', 'HFEN', 'OZON', 'NGMS', 'RTPZ', 'PHIC', 'DGNS', 'DMYI', 'BHSE', 'ABST', 'AJAX', 'GATO', 'ABCM', 'GHLD', 'MCFE', 'MSP', 'XPOA', 'YGMZ', 'BTWN', 'EAR', 'KRBP', 'OPT', 'CDAKQ', 'IPOE', 'IPOF', 'KRON', 'LCY', 'EMPW', 'PACE', 'TPGY', 'APSG', 'ONCR', 'VYGG', 'AGC', 'IMPX', 'ORPHY', 'GRAY', 'SYTA', 'ACTC', 'NMMC', 'VTRU', 'PTVE

In [9]:
import pandas as pd

df = pd.read_csv('../data/cleaned/ipo_master.csv')

missing = ['CIIC', 'HCCO', 'OCFT', 'CHPM', 'ETNB', 'JIH', 'MOHOY', 'CNTG', 'SICP', 'MCMJ', 'OYST', 'TFFP', 'VIE', 'PING', 'IGMS', 'EXPC', 'STSA', 'SWTX', 'CFB', 'CCX', 'WSG', 'FLLC', 'LVGO', 'NOVA', 'PROS', 'MDLA', 'AMK', 'PIC', 'SCPE', 'THCA', 'KRTX', 'CHNG', 'MORF', 'LINX', 'AKRO', 'BCEL', 'PRVL', 'SMMC', 'WORK', 'MWK', 'GIX', 'RTLR', 'AGBA', 'APLT', 'AXLA', 'HHR', 'LCA', 'ATIF', 'SCPL', 'MNRL', 'TPTX', 'TUFN', 'NGM', 'SILK', 'RUHN', 'GNFT', 'SWAV', 'THCB', 'DPHC', 'SOLY', 'MITO', 'AVDR', 'TCRR', 'ANCN', 'HARP', 'GMHI', 'MDJH', 'GBS', 'VII', 'MDWT', 'SCOA', 'VIRI', 'CCV', 'KINZ', 'GHVI', 'MOTV', 'PCPC', 'PTIC', 'SBTX', 'SGTX', 'HTPA', 'KNTE', 'CAP', 'HFEN', 'OZON', 'NGMS', 'RTPZ', 'PHIC', 'DGNS', 'DMYI', 'BHSE', 'ABST', 'AJAX', 'GATO', 'ABCM', 'GHLD', 'MCFE', 'MSP', 'XPOA', 'YGMZ', 'BTWN', 'EAR', 'KRBP', 'OPT', 'CDAKQ', 'IPOE', 'IPOF', 'KRON', 'LCY', 'EMPW', 'PACE', 'TPGY', 'APSG', 'ONCR', 'VYGG', 'AGC', 'IMPX', 'ORPHY', 'GRAY', 'SYTA', 'ACTC', 'NMMC', 'VTRU', 'PTVE', 'RTP', 'SUMO', 'ENPC', 'MTCR', 'LEAP', 'TWCT', 'NSH', 'CRHC', 'HEC', 'CMLF', 'AUVIQ', 'HCDIQ', 'AONE', 'FIII', 'STPK', 'CVAC', 'DCT', 'DGNR', 'DMYD', 'FSDC', 'CMPI', 'FRLN', 'GRSV', 'OSH', 'HOL', 'PRPB', 'VSTA', 'ALVR', 'CCIV', 'INZY', 'ITOS', 'JAMF', 'PSTH', 'CELL', 'PAND', 'TIG', 'DEH', 'HPX', 'PSTX', 'BLCT', 'ACCD', 'DNB', 'AKUS', 'FUSN', 'FMTX', 'GTH', 'NUZE', 'RPTX', 'AZEK', 'GBIO', 'AMTI', 'CALT', 'DADA', 'ZI', 'NARI', 'BMRG', 'NOVS', 'GIK', 'AYLA', 'CLEU', 'GAN', 'IPOB', 'PCPL', 'IPOC', 'DMYT', 'CCXX', 'ZGYH', 'PFHD', 'CSPR', 'PPD', 'ONEM', 'FRES', 'GHIV', 'SCVX', 'DNK', 'ADRT', 'BNOX', 'HCP', 'HORI', 'GLLI', 'MTVC', 'STET', 'LGTO', 'AHI', 'BLEU', 'NETC', 'USER', 'ENCP', 'MYNA', 'LGVC', 'WBEV', 'APNC', 'CIAN', 'JUN', 'ARCK', 'HRT', 'DTC', 'PCCT', 'INFA', 'ENFN', 'GOGN', 'AXH', 'SDIG', 'FEXD', 'AVHI', 'FNA', 'AVDX', 'ISO', 'LCW', 'TCN', 'THRX', 'DMYS', 'EXAI', 'TDCX', 'ARTE', 'HCVI', 'MEKA', 'ESMT', 'SOVO', 'THRN', 'ARGU', 'HHGC', 'FORG', 'CIIG', 'DICE', 'EZFL', 'TWKS', 'FHLT', 'CENQ', 'SSBK', 'PONO', 'ELYM', 'WEBR', 'ICVX', 'MLNK', 'PWSC', 'SNPO', 'OB', 'BASE', 'INST', 'CLOE', 'BRDG', 'IMGO', 'CORS', 'FICV', 'AVTE', 'DNAA', 'DNAB', 'DNAC', 'DNAD', 'IAS', 'THCP', 'EOCW', 'ELEV', 'MFLTF', 'MIRO', 'NEUE', 'AMAM', 'CYT', 'VERV', 'CNVY', 'WKME', 'GSQB', 'LITT', 'ISAA', 'ZMENY', 'OMIC', 'DYNS', 'PSPC', 'POND', 'ORIA', 'BRIV', 'OG', 'TALS', 'BSKY', 'EDR', 'TUGC', 'AGTI', 'IMPLQ', 'KNBE', 'ZY', 'TRKAQ', 'AKYA', 'YTPG', 'ADF', 'RPHM', 'TPGS', 'VECT', 'TIOA', 'CMLT', 'ACHL', 'TWOA', 'IKNA', 'DSEY', 'LCA', 'LVTX', 'OLK', 'VZIO', 'CRZN', 'ACTD', 'DGNU', 'GGPI', 'LEGA', 'RKTA', 'GGMC', 'LDHA', 'LVRA', 'FMIV', 'NAPA', 'RACB', 'VEI', 'GAMC', 'KSI', 'OLO', 'TETC', 'RTPY', 'GTPA', 'GTPB', 'JOANQ', 'LBPH', 'RXDX', 'FHS', 'AGGR', 'DTOC', 'SBII', 'ACQR', 'SVFB', 'SVFC', 'DMYQ', 'FRSG', 'IPVF', 'IPVI', 'WPCA', 'WPCB', 'MACC', 'AMPI', 'DHBC', 'NSTC', 'NSTD', 'IBER', 'LIII', 'SBEA', 'GIIX', 'SCR', 'COLI', 'HIII', 'GSEV', 'BRPM', 'FSII', 'CCVI', 'CHAA', 'CVII', 'DBTX', 'TSIB', 'APR', 'SGFY', 'APGB', 'BPTS', 'SCOB', 'VLON', 'ENNV', 'PICC', 'GSQD', 'TBCP', 'ATC', 'LABP', 'KRNL', 'PRPC', 'TIXT', 'HMPT', 'MIT', 'NLSP', 'OCDX', 'XM', 'FSSI', 'BTNB', 'DHHC', 'HCII', 'NSTB', 'JCIC', 'OEPW', 'MYTE', 'GMBT', 'GMII', 'LEGO', 'CLAS', 'HCCC', 'HCIC', 'TBA', 'POSH', 'FINM', 'HLAH', 'PNTM', 'ENFA', 'KUKE', 'EPWR', 'GRCL', 'LHC', 'PACX', 'PRSR', 'SVFA', 'SWBK', 'AGCB', 'VCKA', 'PPGH', 'STPC', 'THRD', 'STBX', 'CHG', 'FRZA', 'PTWO', 'BRSH', None, 'SKGR', 'PFHC', 'HLVX', 'NUBI', 'MHUA', 'GENQ', 'JGGC', 'GHIX', 'CINC', 'VIGL', 'CRGX', 'RYZB', 'SPGC', 'SRM', 'LQR', 'HRYU', 'PXDT', 'JNVR', 'WRNT', 'TSBX', 'NETD', 'PWM', 'SGE', 'SLRN', 'SYT', 'SBXC', 'DIST', 'PTHR', 'BREA', 'MGOL', 'DYNX', 'SAG', 'CEP', 'BLMZ', 'ZK', 'JUNE', 'CHRO', 'CCCM', 'CCCX', 'MTSR']

df_validated = df[~df['ticker'].isin(missing)]
df_validated = df_validated.dropna(subset=['ticker'])

print(f'Before: {len(df)} rows')
print(f'After removing missing tickers: {len(df_validated)} rows')

df_validated.to_csv('../data/cleaned/ipo_master_validated.csv', index=False)
print('Saved to data/cleaned/ipo_master_validated.csv')

Before: 1675 rows
After removing missing tickers: 1198 rows
Saved to data/cleaned/ipo_master_validated.csv


In [1]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

# test with Snowflake (SNOW) - IPO'd 2020
cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

print('Keys in response:', list(data.keys()))
print('Keys in filings:', list(data['filings'].keys()))
print('')

# check recent filings
filings = data['filings']['recent']
forms = filings['form']
print(f'Total recent filings: {len(forms)}')
print(f'Unique form types: {set(forms)}')
print(f'S-1 in recent: {"S-1" in forms}')

Keys in response: ['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings']
Keys in filings: ['recent', 'files']

Total recent filings: 1019
Unique form types: {'SC 13G/A', '5', 'SC 13G', 'SCHEDULE 13G', '144/A', '144', 'D', '3', '8-K', 'S-8', 'ARS', 'DEFA14A', 'DEF 14A', '4/A', 'SCHEDULE 13G/A', 'PRE 14A', '10-K', '3/A', '4', '10-Q'}
S-1 in recent: False


In [2]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

# check the files array
files = data['filings']['files']
print(f'Number of filing files: {len(files)}')
for f in files:
    print(f)

Number of filing files: 1
{'name': 'CIK0001640147-submissions-001.json', 'filingCount': 106, 'filingFrom': '2017-04-11', 'filingTo': '2021-03-01'}


In [3]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedantt17@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK0001640147-submissions-001.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

forms = data['filingIndex']['form'] if 'filingIndex' in data else data['form']
print(f'Forms in older file: {set(forms)}')
print(f'S-1 found: {"S-1" in forms}')

# find the accession number
for i, form in enumerate(forms):
    if form == 'S-1':
        accessions = data['filingIndex']['accessionNumber'] if 'filingIndex' in data else data['accessionNumber']
        dates = data['filingIndex']['filingDate'] if 'filingIndex' in data else data['filingDate']
        print(f'Accession: {accessions[i]}')
        print(f'Date: {dates[i]}')
        break

Forms in older file: {'SC 13G/A', 'DRSLTR', 'UPLOAD', 'CERT', 'CORRESP', 'SC 13G', '8-A12B', 'D', '3', 'EFFECT', '8-K', 'DRS/A', 'S-1/A', 'S-1', 'S-8', 'D/A', '40-APP/A', '424B4', 'DRS', '3/A', '4', '10-Q', '40-APP'}
S-1 found: True
Accession: 0001628280-20-013010
Date: 2020-08-24


In [4]:
# Written by V
import requests

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research vedant17tiwari@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
}

# test with Snowflake
cik = '0001640147'
url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
data = r.json()

# check older file
f = data['filings']['files'][0]
file_url = f'https://data.sec.gov/submissions/{f["name"]}'
r2 = requests.get(file_url, headers=HEADERS)
print(f'Status: {r2.status_code}')

older = r2.json()
forms = older.get('form', [])
print(f'S-1 in older file: {"S-1" in forms}')

Status: 200
S-1 in older file: True


In [5]:
# Written by V
import requests
import pandas as pd

HEADERS = {
    'User-Agent': 'UC Davis MSBA Research yourucd@ucdavis.edu',
    'Accept-Encoding': 'gzip, deflate',
}

# load your actual data
df = pd.read_csv('../data/cleaned/ticker_cik_map.csv')
df = df[df['cik'].notna()]

# test the first ticker
row = df.iloc[0]
print(f'Testing: {row["ticker"]} -- CIK: {row["cik"]}')

cik = str(row['cik']).zfill(10)
# remove decimal if present
cik = cik.split('.')[0].zfill(10)
print(f'CIK formatted: {cik}')

url = f'https://data.sec.gov/submissions/CIK{cik}.json'
r = requests.get(url, headers=HEADERS)
print(f'Status: {r.status_code}')

if r.status_code == 200:
    data = r.json()
    forms = data['filings']['recent']['form']
    print(f'S-1 in recent: {"S-1" in forms}')
    print(f'Files array length: {len(data["filings"].get("files", []))}')

Testing: INDO -- CIK: 1757840.0
CIK formatted: 0001757840
Status: 200
S-1 in recent: False
Files array length: 0


In [6]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# pick any file that exists
files = os.listdir('../data/raw/edgar/')
print(f'Total cached files: {len(files)}')

# look at the first one
ticker = files[0].replace('.html', '')
print(f'Inspecting: {ticker}')

with open(f'../data/raw/edgar/{files[0]}', 'r', encoding='utf-8', errors='ignore') as f:
    html = f.read()

soup = BeautifulSoup(html, 'lxml')
text = soup.get_text(' ')

# print first 3000 characters to see the structure
print(text[:3000])

Total cached files: 1060
Inspecting: AAPG

 
 
 
 
 
 
 
 
 
 
 SEC.gov | Home 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
      Skip to search field
     
 
      Skip to main content
     
 
 
 
 
 
 
 
 
 
 
 
 An official website of the United States government 
 Here’s how you know 
 
 
 Here’s how you know 
 
 
 
 
 
 
 
 
 
 
                Official websites use .gov
               
 
                              A  .gov  website belongs to an official government organization in the United States.
                           
 
 
 
 
 
 
 
                Secure .gov websites use HTTPS
               
     
              A  lock 
              ( Lock A locked padlock )
              or  https://  means you’ve safely connected to the .gov website. Share sensitive information only on official, secure websites.
             
 
 
 
 
 
 
 
 
 
 
 
 
 SEC homepage 
 
 
 
 
 
        Menu
         
 
 
 
 
 
 
 
 
 
 
          Close
           
 
 
 
 
 Search SEC.gov & EDGAR 
 
 
 
 
 Sea

In [7]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
sec_homepage = 0
real_s1 = 0

for fname in files[:100]:  # check first 100
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    if 'Skip to search field' in html or 'SEC homepage' in html:
        sec_homepage += 1
    else:
        real_s1 += 1

print(f'Checked 100 files')
print(f'Real S-1 documents: {real_s1}')
print(f'SEC homepage (wrong): {sec_homepage}')

Checked 100 files
Real S-1 documents: 14
SEC homepage (wrong): 86


In [8]:
# Written by V
import requests
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

HEADERS = {
    'User-Agent': 'UC Davis Research vedant17tiwari@gmail.com',
    'Accept-Encoding': 'gzip, deflate',
}

# test with Snowflake which we know has a valid S-1
df = pd.read_csv('../data/cleaned/s1_accessions.csv')
snow = df[df['ticker'] == 'SNOW'].iloc[0]

cik_int = int(float(snow['cik']))
accession = snow['accession_number']
acc_nodash = accession.replace('-', '')

index_url = f'https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_nodash}/{accession}-index.htm'
print(f'Index URL: {index_url}')

r = requests.get(index_url, headers={**HEADERS, 'Host': 'www.sec.gov'})
print(f'Status: {r.status_code}')

soup = BeautifulSoup(r.text, 'lxml')
print('\nAll links in index:')
for link in soup.find_all('a', href=True)[:20]:
    print(link['href'])

Index URL: https://www.sec.gov/Archives/edgar/data/1640147/000162828020013010/0001628280-20-013010-index.htm
Status: 200

All links in index:
https://www.sec.gov
https://www.sec.gov
//www.sec.gov/submit-filings/about-edgar
/cgi-bin/browse-edgar?action=getcurrent
https://www.sec.gov/edgar/search-and-access
/index.htm
/edgar/searchedgar/companysearch.html
/Archives/edgar/data/1640147/000162828020013010/snowflakes-1.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit11.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit31.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit33.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit101.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit102.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit103.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit104.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit105.htm
/Archives/edgar/data/1640147/000162828020013010/exhibit1010.htm
/Archives/edga

In [9]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
sec_homepage = 0
real_s1 = 0

for fname in files[:100]:
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    if 'Skip to search field' in html or 'SEC homepage' in html:
        sec_homepage += 1
    else:
        real_s1 += 1

print(f'Checked 100 files')
print(f'Real S-1 documents: {real_s1}')
print(f'SEC homepage (wrong): {sec_homepage}')

Checked 100 files
Real S-1 documents: 100
SEC homepage (wrong): 0


In [10]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# find a file we know has founding year info
files = os.listdir('../data/raw/edgar/')

for fname in files[10:15]:
    ticker = fname.replace('.html', '')
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    
    soup = BeautifulSoup(html, 'lxml')
    text = soup.get_text(' ')
    
    # find sentences mentioning incorporation or founding
    sentences = text.split('.')
    for s in sentences:
        if any(word in s.lower() for word in ['incorporat', 'founded', 'organized', 'formed', 'established']):
            if any(str(y) in s for y in range(2000, 2026)):
                print(f'\n{ticker}:')
                print(s.strip()[:300])
                break


ACLX:
Corporate Information  
 We were incorporated in Delaware in December 2014 under the name Encarta Therapeutics, Inc

ACRV:
Corporate Information    We
were incorporated under the laws of the State of Delaware in March 2018

ACVA:
Corporate Information    We were incorporated
in Delaware in December 2014

ACXP:
However, our annual report on  Form 10-K for the year ended December 31, 2024, filed on March 17, 2025 , and all other documents incorporated by reference into this prospectus that were filed prior to August 4, 2025, do not give effect to the Reverse Stock Split


In [11]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

files = os.listdir('../data/raw/edgar/')
for fname in files[20:25]:
    ticker = fname.replace('.html', '')
    with open(f'../data/raw/edgar/{fname}', 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    soup = BeautifulSoup(html, 'lxml')
    text = soup.get_text(' ')
    sentences = text.split('.')
    for s in sentences:
        if 'raised' in s.lower() or 'funding' in s.lower() or 'series' in s.lower():
            if any(str(y) in s for y in range(2010, 2026)):
                print(f'\n{ticker}: {s.strip()[:300]}')
                break


AERO: In order to facilitate the DGIEs ability to monitor compliance in the future with certain restrictions on foreign ownership set forth in
Mexican law, as described in RegulationRegulation of the Mexican Airline IndustryForeign Investment Limitations under Mexican Law and Description of Capital

AFRM: ​ 
 ​ 
 ​ 
 
 
 Fiscal Year Ended 
 
 June 30, 
 
 
 ​ 
 ​ 
 
 
 Three Months Ended 
 
 September 30, 
 
 
 ​ 
 
 
 ​ 
 ​ 
 ​ 
 
 
 2019  
 
 
 ​ 
 ​ 
 
 
 2020 
 
 
 ​ 
 ​ 
 
 
 2019  
 
 
 ​ 
 ​ 
 
 
 2020 
 
 
 ​ 
 
 
 ​ 
 ​ 
 ​ 
 
 
 (in thousands) 
 
 
 ​ 
 
 
 
 Consolidated Statements of Oper

AFYA: As of September 30, 2019, our learning tools consisted of more than 2,033 video classes, 824 book chapters, 1,270 podcasts, 1,070 summarized texts and an exam
bank of approximately 1,604 questions;  
                Assessment tool :    Broad database suite comprising approximately 85 thousand quizz

AGCC: Type 
 
   
 
 Name 
 
   
 
 Issuing  Authority/ Registration  Insti

In [12]:
# Written by V
import pandas as pd
import sqlite3

conn = sqlite3.connect('../database/ipo_tracker.db')
vc = pd.read_csv('../data/cleaned/vc_history.csv')
print(f'Loading {len(vc)} vc_history rows...')

loaded = 0
for _, row in vc.iterrows():
    try:
        conn.execute('''INSERT OR REPLACE INTO vc_history
            (ticker, founding_year, num_rounds, last_round_type,
             total_funding_usd, has_crunchbase)
            VALUES (?,?,?,?,?,?)''',
            (row['ticker'], row.get('founding_year'), row.get('num_rounds'),
             row.get('last_round_type'), row.get('total_funding_usd'), 0))
        loaded += 1
    except Exception as e:
        print(f'Error {row["ticker"]}: {e}')

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM vc_history').fetchone()[0]
print(f'vc_history table: {count} rows')
conn.close()

Loading 1041 vc_history rows...
vc_history table: 1036 rows


In [13]:
# Written by V
import pandas as pd

df = pd.read_csv('../data/cleaned/price_performance.csv')
print(f'Total rows: {len(df)}')
print(f'Empty rows (no returns): {df["return_90d"].isna().sum()}')
print(f'Suspicious first_day_pop (>100): {(df["first_day_pop"].abs() > 100).sum()}')
print(f'Normal rows: {((df["first_day_pop"].abs() <= 100) & df["return_90d"].notna()).sum()}')

Total rows: 770
Empty rows (no returns): 68
Suspicious first_day_pop (>100): 122
Normal rows: 581


In [14]:
# Written by V
import pandas as pd

df = pd.read_csv('../data/cleaned/price_performance.csv')

# flag suspicious rows but don't drop them
# first_day_pop issues come from bad offer prices, not bad price data
# return values are still valid even when first_day_pop is wrong

# only drop rows where return_90d is missing since that's our target variable
df_clean = df[df['return_90d'].notna()].copy()

# cap first_day_pop at reasonable range for flagging purposes
df_clean['first_day_pop_suspicious'] = (df_clean['first_day_pop'].abs() > 100).astype(int)

print(f'Clean rows with return_90d: {len(df_clean)}')
print(f'Of which first_day_pop suspicious: {df_clean["first_day_pop_suspicious"].sum()}')
print(f'return_90d range: {df_clean["return_90d"].min():.2f} to {df_clean["return_90d"].max():.2f}')

df_clean.to_csv('../data/cleaned/price_performance_clean.csv', index=False)
print('Saved to data/cleaned/price_performance_clean.csv')

Clean rows with return_90d: 702
Of which first_day_pop suspicious: 121
return_90d range: -0.98 to 9.00
Saved to data/cleaned/price_performance_clean.csv


In [15]:
# Written by V
import pandas as pd
import sqlite3

conn = sqlite3.connect('../database/ipo_tracker.db')

# clear existing price_performance table first
conn.execute('DELETE FROM price_performance')
conn.commit()
print('Cleared old price_performance data')

df = pd.read_csv('../data/cleaned/price_performance_clean.csv')
loaded = 0

for _, row in df.iterrows():
    try:
        conn.execute('''INSERT OR REPLACE INTO price_performance
            (ticker, open_day_price, first_day_pop, return_1d,
             return_30d, return_90d, return_180d)
            VALUES (?,?,?,?,?,?,?)''',
            (row['ticker'], row['open_day_price'], row['first_day_pop'],
             row['return_1d'], row['return_30d'], row['return_90d'],
             row['return_180d']))
        loaded += 1
    except Exception as e:
        print(f'Error {row["ticker"]}: {e}')

conn.commit()
count = conn.execute('SELECT COUNT(*) FROM price_performance').fetchone()[0]
print(f'price_performance table: {count} rows loaded')
conn.close()

Cleared old price_performance data
price_performance table: 702 rows loaded


In [16]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

# fix exchange - it's already in ipo_master_validated.csv
master = pd.read_csv('../data/cleaned/ipo_master_validated.csv')
print(f'Exchange values in master: {master["exchange"].value_counts() if "exchange" in master.columns else "column not found"}')
print(f'Columns in master: {list(master.columns)}')

Exchange values in master: column not found
Columns in master: ['ticker', 'company_name', 'ipo_date', 'year']


In [17]:
# Written by V
import pandas as pd

raw = pd.read_csv('../data/raw/ipo_master_raw.csv')
print(f'Columns in raw: {list(raw.columns)}')
print(raw.head(3))

Columns in raw: ['IPO Date', 'Symbol', 'Company Name', 'IPO Price', 'Current', 'Return', 'year']
     IPO Date Symbol                          Company Name IPO Price Current  \
0  2019-12-30  MKDTY                   Molecular Data Inc.     $5.38       -   
1  2019-12-19   INDO  Indonesia Energy Corporation Limited    $11.00   $4.59   
2  2019-12-19   MNPR             Monopar Therapeutics Inc.     $8.00  $60.36   

     Return  year  
0  -100.00%  2019  
1   -58.82%  2019  
2   654.50%  2019  


In [18]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

# fix financials - drop gross_margin and revenue_growth
conn.executescript('''
    CREATE TABLE financials_new (
        ticker TEXT PRIMARY KEY REFERENCES ipos(ticker),
        revenue_ttm REAL,
        net_income_ttm REAL,
        is_profitable INTEGER
    );
    INSERT INTO financials_new 
    SELECT ticker, revenue_ttm, net_income_ttm, is_profitable 
    FROM financials;
    DROP TABLE financials;
    ALTER TABLE financials_new RENAME TO financials;
''')

# fix ipos - drop exchange, total_raise_usd, sector, use_of_proceeds
conn.executescript('''
    CREATE TABLE ipos_new (
        ticker TEXT PRIMARY KEY,
        company_name TEXT NOT NULL,
        ipo_date TEXT NOT NULL,
        offer_price REAL,
        shares_offered INTEGER,
        risk_factors_text TEXT
    );
    INSERT INTO ipos_new
    SELECT ticker, company_name, ipo_date, offer_price, 
           shares_offered, risk_factors_text
    FROM ipos;
    DROP TABLE ipos;
    ALTER TABLE ipos_new RENAME TO ipos;
''')

# fix vc_history - drop lead_investor
conn.executescript('''
    CREATE TABLE vc_history_new (
        ticker TEXT PRIMARY KEY REFERENCES ipos(ticker),
        total_funding_usd REAL,
        num_rounds INTEGER,
        last_round_type TEXT,
        founding_year INTEGER,
        has_crunchbase INTEGER DEFAULT 1
    );
    INSERT INTO vc_history_new
    SELECT ticker, total_funding_usd, num_rounds, 
           last_round_type, founding_year, has_crunchbase
    FROM vc_history;
    DROP TABLE vc_history;
    ALTER TABLE vc_history_new RENAME TO vc_history;
''')

conn.commit()

# verify new null rates
print('New null rates after cleanup:')
for table in ['ipos', 'financials', 'vc_history']:
    df = pd.read_sql(f'SELECT * FROM {table}', conn)
    print(f'\n=== {table} ===')
    print((df.isnull().mean() * 100).round(1).to_string())

conn.close()

New null rates after cleanup:

=== ipos ===
ticker                0.0
company_name          0.0
ipo_date              0.0
offer_price          35.3
shares_offered       91.3
risk_factors_text    23.0

=== financials ===
ticker             0.0
revenue_ttm        1.8
net_income_ttm    30.7
is_profitable      0.0

=== vc_history ===
ticker                0.0
total_funding_usd    69.2
num_rounds           65.0
last_round_type      65.0
founding_year        49.3
has_crunchbase        0.0


In [19]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

df = pd.read_sql('SELECT ticker, offer_price, shares_offered FROM ipos', conn)
df['total_raise_usd'] = df['offer_price'] * df['shares_offered']

derivable = df['total_raise_usd'].notna().sum()
print(f'total_raise_usd derivable for: {derivable} companies')
print(f'Coverage: {derivable/len(df):.1%}')
print(df['total_raise_usd'].describe())

total_raise_usd derivable for: 98 companies
Coverage: 8.2%
count    9.800000e+01
mean     2.038365e+08
std      4.854500e+08
min      8.888890e+01
25%      1.377000e+07
50%      8.114500e+07
75%      1.972600e+08
max      3.666750e+09
Name: total_raise_usd, dtype: float64


In [20]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

# check 1: tickers in ipos missing from price_performance
ipos = pd.read_sql('SELECT ticker FROM ipos', conn)
prices = pd.read_sql('SELECT ticker FROM price_performance', conn)
missing_prices = set(ipos['ticker']) - set(prices['ticker'])
print(f'Tickers in ipos missing from price_performance: {len(missing_prices)}')

# check 2: orphan records in underwriters
orphan_uw = pd.read_sql('''
    SELECT COUNT(*) as count FROM underwriters u
    WHERE u.ticker NOT IN (SELECT ticker FROM ipos)
''', conn).iloc[0,0]
print(f'Orphan underwriter records: {orphan_uw}')

# check 3: offer_price sanity check
offer_prices = pd.read_sql('SELECT offer_price FROM ipos WHERE offer_price IS NOT NULL', conn)
print(f'offer_price range: ${offer_prices["offer_price"].min():.2f} to ${offer_prices["offer_price"].max():.2f}')
suspicious_prices = (offer_prices['offer_price'] < 1).sum()
print(f'offer_price below $1 (suspicious): {suspicious_prices}')

# check 4: return_90d range
returns = pd.read_sql('SELECT return_90d FROM price_performance WHERE return_90d IS NOT NULL', conn)
print(f'return_90d range: {returns["return_90d"].min():.2f} to {returns["return_90d"].max():.2f}')
extreme = (returns['return_90d'].abs() > 5).sum()
print(f'return_90d beyond +500% or -100%: {extreme}')

conn.close()

Tickers in ipos missing from price_performance: 489
Orphan underwriter records: 0
offer_price range: $0.00 to $700.00
offer_price below $1 (suspicious): 129
return_90d range: -0.98 to 9.00
return_90d beyond +500% or -100%: 2


In [21]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

conn.executescript('''
    DROP VIEW IF EXISTS ipo_summary;
    DROP VIEW IF EXISTS sector_performance;
    DROP VIEW IF EXISTS underwriter_league;

    CREATE VIEW ipo_summary AS
    SELECT
        i.ticker, i.company_name, i.ipo_date,
        i.offer_price, i.shares_offered, i.risk_factors_text,
        f.revenue_ttm, f.net_income_ttm, f.is_profitable,
        v.total_funding_usd, v.num_rounds, v.last_round_type,
        v.founding_year, v.has_crunchbase,
        p.open_day_price, p.first_day_pop,
        p.return_1d, p.return_30d, p.return_90d, p.return_180d
    FROM ipos i
    LEFT JOIN financials f ON i.ticker = f.ticker
    LEFT JOIN vc_history v ON i.ticker = v.ticker
    LEFT JOIN price_performance p ON i.ticker = p.ticker;

    CREATE VIEW underwriter_league AS
    SELECT
        u.underwriter_name,
        COUNT(DISTINCT u.ticker) AS deal_count,
        SUM(u.is_lead) AS lead_count,
        ROUND(AVG(p.first_day_pop) * 100, 2) AS avg_first_day_pop_pct,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct
    FROM underwriters u
    LEFT JOIN price_performance p ON u.ticker = p.ticker
    GROUP BY u.underwriter_name
    ORDER BY deal_count DESC;

    CREATE VIEW ipo_by_year AS
    SELECT
        strftime('%Y', ipo_date) AS year,
        COUNT(*) AS ipo_count,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct,
        ROUND(AVG(p.first_day_pop) * 100, 2) AS avg_first_day_pop_pct
    FROM ipos i
    LEFT JOIN price_performance p ON i.ticker = p.ticker
    GROUP BY year
    ORDER BY year;
''')

conn.commit()
print('Views created successfully')

# verify each view
print('\nipo_summary sample:')
print(pd.read_sql('SELECT ticker, company_name, return_90d FROM ipo_summary WHERE return_90d IS NOT NULL LIMIT 5', conn))

print('\nunderwriter_league top 10:')
print(pd.read_sql('SELECT * FROM underwriter_league LIMIT 10', conn))

print('\nipo_by_year:')
print(pd.read_sql('SELECT * FROM ipo_by_year', conn))

conn.close()

Views created successfully

ipo_summary sample:
  ticker               company_name  return_90d
0   MNPR  Monopar Therapeutics Inc.     -0.7223
1   BILL        BILL Holdings, Inc.      0.3149
2     XP                    XP Inc.     -0.2449
3    CAN                Canaan Inc.     -0.6730
4   SITM         SiTime Corporation      0.0654

underwriter_league top 10:
  underwriter_name  deal_count  lead_count  avg_first_day_pop_pct  \
0              UBS         966         439           8.498785e+11   
1    Goldman Sachs         283         283           4.010377e+06   
2   Morgan Stanley         280         129           2.370121e+06   
3      J.P. Morgan         239          46           5.727844e+06   
4  BofA Securities         190           0           3.681155e+05   
5        Citigroup         164          28           2.193364e+06   
6    Credit Suisse         146          17           3.472303e+06   
7        Jefferies         143           0           2.891844e+06   
8         Barcl

In [22]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

conn.executescript('''
    DROP VIEW IF EXISTS ipo_summary;
    DROP VIEW IF EXISTS underwriter_league;
    DROP VIEW IF EXISTS ipo_by_year;

    CREATE VIEW ipo_summary AS
    SELECT
        i.ticker, i.company_name, i.ipo_date,
        i.offer_price, i.shares_offered, i.risk_factors_text,
        f.revenue_ttm, f.net_income_ttm, f.is_profitable,
        v.total_funding_usd, v.num_rounds, v.last_round_type,
        v.founding_year, v.has_crunchbase,
        p.open_day_price,
        CASE WHEN ABS(p.first_day_pop) > 100 THEN NULL 
             ELSE p.first_day_pop END AS first_day_pop,
        p.return_1d, p.return_30d, p.return_90d, p.return_180d
    FROM ipos i
    LEFT JOIN financials f ON i.ticker = f.ticker
    LEFT JOIN vc_history v ON i.ticker = v.ticker
    LEFT JOIN price_performance p ON i.ticker = p.ticker;

    CREATE VIEW underwriter_league AS
    SELECT
        u.underwriter_name,
        COUNT(DISTINCT u.ticker) AS deal_count,
        SUM(u.is_lead) AS lead_count,
        ROUND(AVG(CASE WHEN ABS(p.first_day_pop) <= 100 
                       THEN p.first_day_pop END) * 100, 2) AS avg_first_day_pop_pct,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct
    FROM underwriters u
    LEFT JOIN price_performance p ON u.ticker = p.ticker
    GROUP BY u.underwriter_name
    ORDER BY deal_count DESC;

    CREATE VIEW ipo_by_year AS
    SELECT
        strftime('%Y', ipo_date) AS year,
        COUNT(*) AS ipo_count,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct,
        ROUND(AVG(CASE WHEN ABS(p.first_day_pop) <= 100 
                       THEN p.first_day_pop END) * 100, 2) AS avg_first_day_pop_pct
    FROM ipos i
    LEFT JOIN price_performance p ON i.ticker = p.ticker
    GROUP BY year
    ORDER BY year;
''')

conn.commit()
print('Views updated')

print('\nunderwriter_league top 10:')
print(pd.read_sql('SELECT * FROM underwriter_league LIMIT 10', conn))

print('\nipo_by_year:')
print(pd.read_sql('SELECT * FROM ipo_by_year', conn))

conn.close()

Views updated

underwriter_league top 10:
  underwriter_name  deal_count  lead_count  avg_first_day_pop_pct  \
0              UBS         966         439                1148.20   
1    Goldman Sachs         283         283                1329.52   
2   Morgan Stanley         280         129                1323.64   
3      J.P. Morgan         239          46                1221.15   
4  BofA Securities         190           0                1108.37   
5        Citigroup         164          28                1179.44   
6    Credit Suisse         146          17                 921.40   
7        Jefferies         143           0                1236.93   
8         Barclays         131          14                1211.75   
9            Cowen         117           0                1524.75   

   avg_90d_return_pct  
0               -8.01  
1                1.19  
2               -1.36  
3                1.81  
4                1.97  
5                3.92  
6                6.81  
7     

In [23]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

df = pd.read_sql('SELECT first_day_pop FROM price_performance WHERE first_day_pop IS NOT NULL', conn)

print('first_day_pop distribution:')
print(f'Total rows: {len(df)}')
print(f'Below 2 (200%): {(df["first_day_pop"] < 2).sum()}')
print(f'Between 2 and 10: {((df["first_day_pop"] >= 2) & (df["first_day_pop"] < 10)).sum()}')
print(f'Between 10 and 100: {((df["first_day_pop"] >= 10) & (df["first_day_pop"] < 100)).sum()}')
print(f'Above 100: {(df["first_day_pop"] >= 100).sum()}')
print(f'\nMedian: {df["first_day_pop"].median():.4f}')
print(f'Mean excluding >10: {df[df["first_day_pop"] < 10]["first_day_pop"].mean():.4f}')
print(f'Rows with first_day_pop < 2: {(df["first_day_pop"] < 2).sum()}')

conn.close()

first_day_pop distribution:
Total rows: 702
Below 2 (200%): 277
Between 2 and 10: 122
Between 10 and 100: 182
Above 100: 121

Median: 5.7417
Mean excluding >10: 1.6859
Rows with first_day_pop < 2: 277


In [24]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

conn.executescript('''
    DROP VIEW IF EXISTS ipo_summary;
    DROP VIEW IF EXISTS underwriter_league;
    DROP VIEW IF EXISTS ipo_by_year;

    CREATE VIEW ipo_summary AS
    SELECT
        i.ticker, i.company_name, i.ipo_date,
        i.offer_price, i.shares_offered, i.risk_factors_text,
        f.revenue_ttm, f.net_income_ttm, f.is_profitable,
        v.total_funding_usd, v.num_rounds, v.last_round_type,
        v.founding_year, v.has_crunchbase,
        p.open_day_price,
        p.return_1d, p.return_30d, p.return_90d, p.return_180d
    FROM ipos i
    LEFT JOIN financials f ON i.ticker = f.ticker
    LEFT JOIN vc_history v ON i.ticker = v.ticker
    LEFT JOIN price_performance p ON i.ticker = p.ticker;

    CREATE VIEW underwriter_league AS
    SELECT
        u.underwriter_name,
        COUNT(DISTINCT u.ticker) AS deal_count,
        SUM(u.is_lead) AS lead_count,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct,
        ROUND(AVG(p.return_180d) * 100, 2) AS avg_180d_return_pct
    FROM underwriters u
    LEFT JOIN price_performance p ON u.ticker = p.ticker
    GROUP BY u.underwriter_name
    ORDER BY deal_count DESC;

    CREATE VIEW ipo_by_year AS
    SELECT
        strftime('%Y', ipo_date) AS year,
        COUNT(*) AS ipo_count,
        ROUND(AVG(p.return_90d) * 100, 2) AS avg_90d_return_pct,
        ROUND(AVG(p.return_180d) * 100, 2) AS avg_180d_return_pct
    FROM ipos i
    LEFT JOIN price_performance p ON i.ticker = p.ticker
    GROUP BY year
    ORDER BY year;
''')

conn.commit()
print('Views updated')

print('\nunderwriter_league top 10:')
print(pd.read_sql('SELECT * FROM underwriter_league LIMIT 10', conn))

print('\nipo_by_year:')
print(pd.read_sql('SELECT * FROM ipo_by_year', conn))

conn.close()

Views updated

underwriter_league top 10:
  underwriter_name  deal_count  lead_count  avg_90d_return_pct  \
0              UBS         966         439               -8.01   
1    Goldman Sachs         283         283                1.19   
2   Morgan Stanley         280         129               -1.36   
3      J.P. Morgan         239          46                1.81   
4  BofA Securities         190           0                1.97   
5        Citigroup         164          28                3.92   
6    Credit Suisse         146          17                6.81   
7        Jefferies         143           0               12.04   
8         Barclays         131          14                7.13   
9            Cowen         117           0               -3.09   

   avg_180d_return_pct  
0               -18.90  
1                -0.91  
2                -9.30  
3                -2.33  
4                 3.18  
5                 3.90  
6                -1.02  
7                 7.41  
8     

In [25]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')
ubs = pd.read_sql('''
    SELECT ticker, underwriter_name, is_lead 
    FROM underwriters 
    WHERE underwriter_name = "UBS" 
    LIMIT 10
''', conn)
print(ubs)
print(f'\nTotal UBS rows: {pd.read_sql("SELECT COUNT(*) FROM underwriters WHERE underwriter_name = \'UBS\'", conn).iloc[0,0]}')
conn.close()

  ticker underwriter_name  is_lead
0   MNPR              UBS        1
1   BILL              UBS        0
2     EH              UBS        0
3     XP              UBS        0
4    CAN              UBS        0
5   SITM              UBS        0
6   CNSP              UBS        1
7   KRKR              UBS        0
8   TELA              UBS        0
9   GRTX              UBS        0

Total UBS rows: 983


In [26]:
# Written by V
import os
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# check a company that shouldn't have UBS as underwriter
ticker = 'MNPR'
with open(f'../data/raw/edgar/{ticker}.html', 'r', encoding='utf-8', errors='ignore') as f:
    html = f.read()

soup = BeautifulSoup(html, 'lxml')
text = soup.get_text(' ')

# find UBS mentions
import re
sentences = text.split('.')
ubs_mentions = [s.strip()[:200] for s in sentences if 'UBS' in s]
print(f'UBS mentions in {ticker} S-1:')
for m in ubs_mentions[:5]:
    print(f'  - {m}')

UBS mentions in MNPR S-1:


In [27]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

# check if MNPR has duplicate underwriter rows
mnpr = pd.read_sql("SELECT * FROM underwriters WHERE ticker = 'MNPR'", conn)
print('MNPR underwriters:')
print(mnpr)

# check total underwriter rows vs expected
total = pd.read_sql('SELECT COUNT(*) as count FROM underwriters', conn).iloc[0,0]
unique_tickers = pd.read_sql('SELECT COUNT(DISTINCT ticker) as count FROM underwriters', conn).iloc[0,0]
print(f'\nTotal rows: {total}')
print(f'Unique tickers: {unique_tickers}')
print(f'Average underwriters per company: {total/unique_tickers:.1f}')

conn.close()

MNPR underwriters:
   id ticker underwriter_name  is_lead
0   1   MNPR              UBS        1

Total rows: 3631
Unique tickers: 966
Average underwriters per company: 3.8


In [28]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

# remove UBS rows since they are unreliable
before = conn.execute('SELECT COUNT(*) FROM underwriters').fetchone()[0]
conn.execute("DELETE FROM underwriters WHERE underwriter_name = 'UBS'")
conn.commit()
after = conn.execute('SELECT COUNT(*) FROM underwriters').fetchone()[0]

print(f'Removed {before - after} UBS rows')
print(f'Remaining underwriter rows: {after}')

# verify league table now looks sensible
print('\nUpdated underwriter_league top 10:')
print(pd.read_sql('SELECT * FROM underwriter_league LIMIT 10', conn))

conn.close()

Removed 983 UBS rows
Remaining underwriter rows: 2648

Updated underwriter_league top 10:
  underwriter_name  deal_count  lead_count  avg_90d_return_pct  \
0    Goldman Sachs         283         283                1.19   
1   Morgan Stanley         280         129               -1.36   
2      J.P. Morgan         239          46                1.81   
3  BofA Securities         190           0                1.97   
4        Citigroup         164          28                3.92   
5    Credit Suisse         146          17                6.81   
6        Jefferies         143           0               12.04   
7         Barclays         131          14                7.13   
8            Cowen         117           0               -3.09   
9           Stifel         111           0                4.44   

   avg_180d_return_pct  
0                -0.91  
1                -9.30  
2                -2.33  
3                 3.18  
4                 3.90  
5                -1.02  
6       

In [29]:
# Written by V
import sqlite3
import pandas as pd

conn = sqlite3.connect('../database/ipo_tracker.db')

df = pd.read_sql('SELECT * FROM ipo_summary', conn)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Rows with return_90d: {df["return_90d"].notna().sum()}')
print(f'Rows with risk_factors_text: {df["risk_factors_text"].notna().sum()}')
print(f'Rows with revenue_ttm: {df["revenue_ttm"].notna().sum()}')

df.to_csv('../data/cleaned/ipo_analysis_ready.csv', index=False)
print('\nSaved to data/cleaned/ipo_analysis_ready.csv')

conn.close()

Shape: (1191, 19)
Columns: ['ticker', 'company_name', 'ipo_date', 'offer_price', 'shares_offered', 'risk_factors_text', 'revenue_ttm', 'net_income_ttm', 'is_profitable', 'total_funding_usd', 'num_rounds', 'last_round_type', 'founding_year', 'has_crunchbase', 'open_day_price', 'return_1d', 'return_30d', 'return_90d', 'return_180d']
Rows with return_90d: 702
Rows with risk_factors_text: 917
Rows with revenue_ttm: 850

Saved to data/cleaned/ipo_analysis_ready.csv
